# Retrieval from ChromaDB

Queries the ChromaDB collection built by `notebooks/ingestion_preprocessing_embedding.ipynb`.

**Run the ingestion notebook first** so `notebooks/chroma_db/` exists on disk.

In [1]:
!pip install chromadb transformers torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import chromadb
from transformers import AutoTokenizer, AutoModel
import torch

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("actas_obra")
print(f"Collection loaded with {collection.count()} chunks.")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection loaded with 668 chunks.


In [3]:
model_name = "BSC-LT/MrBERT-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 14965.30it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def embed_query(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

In [6]:
search_query = "¿Cuál es el estado actual de la gestión del informe de conexión de acometida pluvial con Drenajes Besós?"

query_embedding = embed_query(search_query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"--- Rank {i+1} | Chunk ID: {meta['chunk_id']} | Distance: {dist:.4f} ---")
    print(doc)
    print()

--- Rank 1 | Chunk ID: 235 | Distance: 524.4314 ---
Para cerrar la verificación de la ejecución de hinca de perfiles, faltaría confirmar profundidades reales de hinca (empotramiento en estratos competentes), separaciones y orientación respecto al talud calculado. PPI. Además, se han de aportar los Certificados de material de hinca (tipo de carril, acero, características). El Anejo de cálculo modela hincas de carril 54E1. AR-30 Planificación actualizada DF solicita a la constructora la planificación actualizada de la obra.

--- Rank 2 | Chunk ID: 114 | Distance: 587.0524 ---
Situación de los bajantes en Documentación Gráfica Se solicita a la constructora la situación de los bajantes de la cubierta correspondientes al área de actuación de obras, en una documentación gráfica, y la definición del diámetro del bajante. AR04- Chapa de remate encuentro canalón y edificio existente La constructora propone no reinstalar la chapa de remate existente entre canalón y edificio existente, ya que pod